# 01 — Environment Setup

Creates the project skeleton, verifies the scientific stack, and writes the
three project-level files: `README.md`, `requirements.txt` and
`environment_report.txt`.

**Run order:** `01` → `02` → `03`.

**Notes**
- Every notebook starts with the same project-root bootstrap (`_dj_root()`).
  It resolves the Google Colab Drive location when available, and otherwise
  the local checkout that contains a `notebooks/` directory.
- Set the `DORJIBARI_ROOT` environment variable to override the detected root.
- Cells run top to bottom: later cells reuse variables defined earlier.
- The Google Drive cell mounts Drive only when running inside Colab; outside
  Colab it is skipped and the local project root is used.
- `requirements.txt` and `environment_report.txt` are regenerated for the
  environment that runs this notebook. The unmodified Colab originals are
  kept in `reference/` and this notebook re-creates that archive before it
  overwrites either top-level file.
- Saved outputs below come from the original Colab run. A repaired cell can
  therefore print a different log locally while producing the same files.


In [13]:
# ======================================================================
# DorjiBari project root bootstrap (identical in all notebooks).
# Resolves the project root in Google Colab or in a local checkout.
# Override with the DORJIBARI_ROOT environment variable if needed.
# ======================================================================
import os

_COLAB_PROJECT_ROOT = "/content/drive/MyDrive/DorjiBari-Research"
_COLAB_DRIVE_ROOT = "/content/drive/MyDrive"


def _dj_mount_drive():
    """Mount Google Drive when running inside Google Colab."""
    if os.path.isdir(_COLAB_DRIVE_ROOT):
        return
    try:
        import importlib
        drive = importlib.import_module("google.colab.drive")
    except ImportError:
        return
    drive.mount("/content/drive")


def _dj_root():
    """Return the absolute path of the DorjiBari-Research project root.

    Resolution order:
      1. the DORJIBARI_ROOT environment variable;
      2. the Google Colab Drive location;
      3. the nearest parent of the working directory containing "notebooks/".

    Raises RuntimeError instead of guessing, so that no notebook writes into
    an unrelated directory.
    """
    _dj_mount_drive()

    env_root = os.environ.get("DORJIBARI_ROOT")
    if env_root:
        if os.path.isdir(os.path.join(env_root, "notebooks")):
            return os.path.abspath(env_root)
        raise RuntimeError(
            "DORJIBARI_ROOT is set to {!r}, but that directory has no "
            "'notebooks' folder. Point it at the DorjiBari-Research project "
            "root.".format(env_root)
        )

    candidates = [_COLAB_PROJECT_ROOT]

    current = os.path.abspath(os.getcwd())
    while True:
        candidates.append(current)
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent

    for candidate in candidates:
        if os.path.isdir(os.path.join(candidate, "notebooks")):
            return os.path.abspath(candidate)

    raise RuntimeError(
        "Could not locate the DorjiBari-Research project root: no parent of "
        "{!r} contains a 'notebooks' folder, and the Google Colab Drive "
        "location is unavailable. Start Jupyter/VS Code from inside the "
        "project, or set DORJIBARI_ROOT to the project root.".format(os.getcwd())
    )


PROJECT_ROOT = _dj_root()
print("DorjiBari project root:", PROJECT_ROOT)



# Dependencies: pip install -r requirements-local.txt
import sys
import platform
import torch
import cv2
import numpy as np
import pandas as pd
import sklearn

print("=" * 55)
print("DORJIBARI RESEARCH - ENVIRONMENT REPORT")
print("=" * 55)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nGPU AVAILABLE:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU NAME:", torch.cuda.get_device_name(0))
    print("CUDA VERSION:", torch.version.cuda)

print("\nENVIRONMENT CHECK COMPLETED")


DorjiBari project root: d:\DorjiBari-Research


Note: you may need to restart the kernel to use updated packages.
DORJIBARI RESEARCH - ENVIRONMENT REPORT
Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26300-SP0
PyTorch: 2.15.0+cpu
OpenCV: 4.14.0
NumPy: 2.5.5
Pandas: 3.1.1
Scikit-learn: 1.9.1

GPU AVAILABLE: False

ENVIRONMENT CHECK COMPLETED


In [12]:

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.randn(1000, 1000, device=device)
y = torch.randn(1000, 1000, device=device)

z = torch.matmul(x, y)

if device.type == "cuda":
    torch.cuda.synchronize()

print("Device:", device)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if device.type == "cuda" else "Not Available (CPU execution)"
)
print("Output Shape:", z.shape)
print("Output Device:", z.device)
print(
    "GPU Computation: SUCCESS"
    if device.type == "cuda" else "CPU Computation: SUCCESS"
)


Device: cpu
GPU: Not Available (CPU execution)
Output Shape: torch.Size([1000, 1000])
Output Device: cpu
CPU Computation: SUCCESS


In [10]:

import os

_dj_mount_drive()

if os.path.isdir(_COLAB_DRIVE_ROOT):
    print("Google Drive Connected Successfully!")
else:
    print("Not running in Google Colab - skipping Google Drive mount.")


Not running in Google Colab - skipping Google Drive mount.


In [9]:

import os

PROJECT_ROOT = _dj_root()

folders = [
    "dataset/BodyM",
    "dataset/Local_Dataset",
    "notebooks",
    "src",
    "configs",
    "experiments/predictions",
    "experiments/metrics",
    "experiments/logs",
    "checkpoints",
    "figures",
    "reports",
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_ROOT, folder), exist_ok=True)

print("Project Structure Created Successfully!\n")

# Keep the listing readable: never descend into the virtual environment,
# VCS metadata, caches or the (potentially huge) preprocessed image tree.
SKIP_DIRS = {
    ".venv", "venv", ".git", ".ipynb_checkpoints", "__pycache__",
    "preprocessed_images",
}
MAX_DEPTH = 2

for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, "").count(os.sep)
    dirs[:] = sorted(d for d in dirs if d not in SKIP_DIRS)
    if level > MAX_DEPTH:
        continue
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")


Project Structure Created Successfully!

DorjiBari-Research/
    checkpoints/
    configs/
    dataset/
        BodyM/
        Local_Dataset/
    experiments/
        logs/
        metrics/
        predictions/
    figures/
    notebooks/
    reference/
        pre-completion-backup/
    reports/
    scripts/
    src/


In [8]:

import os

readme = r"""
# DorjiBari Research

## Project Overview
Smartphone-based body measurement estimation
using front-view image, side-view image,
and measured human height.

## Research Objectives
- Dual-view body measurement regression
- Measurement-specific uncertainty estimation
- Cost-aware accept, retake, and manual verification

## BodyM Target Measurements
1. Chest
2. Waist
3. Hip
4. Shoulder Breadth
5. Arm Length

## Experiments
- E01: Height-only baseline
- E02: Silhouette geometry baseline
- E03: Front vs Side vs Both
- E04: Classical ML comparison
- E05: Uncertainty calibration and tolerance-based selective verification

## Research Rules
- Prevent subject-level data leakage.
- Preserve official dataset splits.
- Never tune using official test sets.
- Save configurations, predictions and metrics.
- Report all measurements in centimetres.

## Dataset
Source: the official BodyM dataset in the public `amazon-bodym` S3 bucket
(region `us-west-2`, unsigned access, no credentials required). The official
Train / TestA / TestB split definitions are preserved.

Expected local layout:

```
dataset/
  BodyM/
    dataset_structure.txt          S3 structure audit
    audit/                         object inventories per split
    metadata/                      source CSVs and QC audit tables
    processed/                     manifests, split-repair backups
    preprocessed_images/
      train/{front,side}/*.png     normalised 256x256 silhouettes
      testA/{front,side}/*.png
      testB/{front,side}/*.png
  Local_Dataset/                   optional local capture images (unused by E01-E05)
```

`dataset/BodyM/` is empty in a fresh checkout. Notebook 02 downloads the
metadata CSVs; notebook 03 downloads the silhouette images and builds the
preprocessed tree (8,975 photo pairs, 17,950 PNG files).

## Environment
### Original environment (Google Colab)
The unmodified Colab records are archived in `reference/`:
`reference/environment_report-colab-original.txt` and
`reference/requirements-colab-original.txt`. They describe:
Python 3.13.16, PyTorch 2.14.1+cu130, OpenCV 4.14.0, NumPy 2.4.6,
Pandas 2.2.3, scikit-learn 1.6.1, Tesla T4 GPU.

The archived freeze is kept for reference only. Do not install it blindly on
another machine.

### Most recent local run
`requirements.txt` and `environment_report.txt` at the project root describe
the most recent notebook 01 run (currently Python 3.14.6, CPU only). Notebook 01
rewrites both files on every run; it creates the `reference/` archive first if
the archive is missing, then never overwrites it. The archive is the Colab
record; the top-level files always describe the latest run.

### Local CPU environment (Python 3.14)
Use `requirements-local.txt`. The pipeline runs on CPU; no CUDA is required.

```
python -m venv .venv
.venv\Scripts\activate          # Windows
source .venv/bin/activate       # macOS / Linux
pip install -r requirements-local.txt
```

Python 3.14 has no wheels for the exact Colab versions of pandas and NumPy,
so the local pins are newer (pandas 3.x). The APIs used by the notebooks are
unchanged, but small formatting-level differences are possible. Use Colab with
the original freeze when byte-level parity with the recorded run matters.

### VS Code / Jupyter
`.vscode/settings.json` selects `.venv/Scripts/python.exe` and sets the
notebook working directory to the project root. If VS Code still shows missing
imports, select the same interpreter as the notebook kernel:
**Python Environments → `.venv`**. The notebooks detect Google Colab through
`importlib`, not a static `import google.colab`, so Pylance does not report an
unresolved Colab import.

## Notebook Execution Order
1. `notebooks/01_environment_setup.ipynb`
2. `notebooks/02_dataset_audit.ipynb`
3. `notebooks/03_Dataset_Preprocessing.ipynb`

Run each notebook top to bottom. Every notebook starts with the same project
root bootstrap (`_dj_root()`), which resolves the project root in this order:

1. the `DORJIBARI_ROOT` environment variable, if set;
2. `/content/drive/MyDrive/DorjiBari-Research` on Google Colab;
3. the nearest parent directory that contains a `notebooks/` folder.

No path editing is required when moving between Colab and a local checkout.
If no location matches, the notebook stops with an explicit error instead of
writing into an unrelated directory.

### Restart and re-run rules
- Notebook 01 is safe to re-run from top to bottom. It rewrites `README.md`,
  `requirements.txt` and `environment_report.txt`, and creates the `reference/`
  Colab archive only if it is missing.
- Notebook 02 is the authoritative producer of the visual-QC tables
  (`problematic_image_audit.csv`, `alternative_photo_pair_visual_review.csv`).
  Notebook 03 contains a restart-safe fallback: if those files already exist,
  it validates and preserves them instead of overwriting them.
- Notebook 03 cells 28-32 are the recorded subject-overlap investigation. They
  temporarily remove the one TestA/TestB shared subject. Cells 33-34 restore the
  official split membership, so do not stop before cell 34 and do not interpret
  split counts from that temporary state.
- Saved notebook outputs are evidence from the original Colab run, except for
  notebooks 01 and 02, which were deliberately re-executed locally on
  Windows/CPU. Their live outputs describe that local run; the Colab outputs
  stay in `reference/pre-completion-backup/`. Notebook 03 still holds the
  original Colab evidence and must not be re-executed without a backup.
- `reference/pre-completion-backup/` keeps the notebooks and documents exactly
  as they were before this repair pass. The live files can therefore be
  compared against a known earlier state.

## Pipeline Stages
| Stage | Notebook | Content | Main outputs |
|---|---|---|---|
| 1 | 01 | Environment check and project skeleton | `README.md`, `requirements.txt`, `environment_report.txt` |
| 2 | 02 | BodyM audit and QC decisions | `dataset/BodyM/metadata/`, `dataset/BodyM/audit/` |
| 3 | 03 | Manifests, split repair, image preprocessing | `dataset/BodyM/processed/`, `preprocessed_images/` |
| 4 | 03 | E01 height-only baseline | `experiments/E01_height_only/` |
| 5 | 03 | E02 silhouette geometry (52 features) | `experiments/E02_geometry/` |
| 6 | 03 | E03 front / side / both ablation | `experiments/E03_view_ablation/` |
| 7 | 03 | E04 Ridge / RandomForest / MLP comparison | `experiments/E04_classical_ml/` |
| 8 | 03 | E05 uncertainty calibration and selective verification | `experiments/E05_uncertainty_calibration/` |
| 9 | 03 | Evidence consolidation | `reports/` |

## Expected Output Locations
- `dataset/BodyM/` - source metadata, QC audits, manifests, preprocessed silhouettes
- `experiments/E01_height_only/` - E01 config, target medians
- `experiments/E02_geometry/` - feature tables, feature audits, configs
- `experiments/E03_view_ablation/` - ablation config and integrity audit
- `experiments/E04_classical_ml/` - CV results and experiment config
- `experiments/E05_uncertainty_calibration/` - OOF predictions, calibration
  thresholds, coverage, associations, nested uncertainty, risk ranking,
  tolerance-based verification tables
- `experiments/metrics/` - E01-E04 metric tables and the consolidated table
- `experiments/predictions/` - E01-E04 prediction tables
- `experiments/logs/` - preprocessing summary, failures, integrity audit
- `checkpoints/` - fitted model bundles (`joblib`)
- `configs/` - preprocessing configuration
- `figures/` - figures recovered from notebook outputs
- `reports/` - E05 evidence report, inventories and ranking tables
- `src/` - reserved for extracted modules; all logic currently lives in notebooks
- `scripts/` - `validate_project.py`, the project validator (no data required)
- `reference/` - unmodified Colab `requirements.txt` and `environment_report.txt`
  archives; never overwritten by notebook runs

## Reproducing Existing Experiments
1. Create the environment (`requirements-local.txt`) or open the notebooks in
   Colab with the original environment.
2. Run notebook 01 to create the folder skeleton.
3. Run notebook 02 to download the BodyM metadata and produce the QC audits.
   This requires network access to `amazon-bodym`.
4. Run notebook 03 top to bottom. Stage 3 downloads and preprocesses 17,950
   silhouette images; Stage 9 runs nested cross-validation for E05.

Long-running stages are not executed automatically. Check
`PROJECT_INVENTORY.md` for which artifacts already exist on disk and which
must be regenerated by re-running.

## Known Limitations and Unresolved Dependencies
- The original Colab output tree is not fully present in this checkout. A
  partial local run has produced the BodyM metadata/audit tables, the manifests
  and the Train silhouettes (Stage 3 is still filling the image tree); the
  TestA/TestB images and the `experiments/` tables are still missing.
  `PROJECT_INVENTORY.md` lists every expected artifact, its producing stage,
  and its current status.
- Figures, the preprocessing config and the E05 evidence report were recovered
  from the notebook sources and saved outputs; numeric prediction and metric
  tables must be regenerated by re-running the pipeline.
- TestA and TestB share one subject ID in the official metadata. Train is
  disjoint from both splits, but TestA and TestB are not fully subject-disjoint.
  This is documented as an E05 limitation and must be reported in any write-up.
- E05 tolerances (+/- cm) are illustrative sensitivity thresholds, not clinical
  or industry standards.
- Recommendation thresholds calibrated on Train undercover on TestB (15 of 15
  target/level combinations), so TestB must not be used to tune thresholds.
- E01 waist behaves differently on TestA and TestB; the height-only Ridge model
  does not beat the training-median baseline on TestA. This is carried into E05
  and is not resolved.
- `dataset/Local_Dataset/` is empty by design; no local capture data exists yet.

## Validation
A lightweight validator checks the repairs without downloading data or
running experiments:

```
python scripts/validate_project.py
```

It verifies notebook JSON/nbformat validity, that every code cell compiles,
that the project-root bootstrap is present, that no Colab-only path remains in
executed code, that the required directories exist, that the recovered figures
and reports parse, that `README.md` matches the notebook 01 template, and that
every third-party import is covered by `requirements-local.txt`.

To additionally check what happened to the saved notebook outputs, point it at
the pre-repair notebook copies (PowerShell):

```
$env:DORJIBARI_NOTEBOOK_BACKUP = "D:\DorjiBari-Research\reference\pre-completion-backup"
python scripts/validate_project.py
```

Notebook 03 holds the experiment evidence, so its saved outputs and execution
counts must match the backup exactly. The only accepted difference there is the
removal of an illegal `metadata` key from two `stream` outputs, which nbformat
rejects; the text is byte-identical. Notebooks 01 and 02 were intentionally
re-executed locally, so their output differences are reported as `INFO` lines
instead of failures; the original Colab outputs stay in the backup.

### Verification status
- `python scripts/validate_project.py` passes on the current checkout
  (`ALL CHECKS PASSED`). With `DORJIBARI_NOTEBOOK_BACKUP` set it prints only
  `INFO` lines for notebooks 01 and 02 (re-executed locally) and a strict
  `PASS` for notebook 03.
- Notebook 01 was executed end-to-end in a fresh isolated kernel: all 8 code
  cells completed, the README template reproduced `README.md` byte for byte,
  and the `reference/` archives were unchanged. OpenCV is imported explicitly,
  so the environment report no longer depends on earlier kernel state.
- Notebook 02 was re-executed on the available BodyM metadata; its saved
  outputs describe that local run.
- Notebook 03 was validated statically (JSON, nbformat, cell compilation,
  dependency coverage) and its saved outputs were preserved byte for byte, but
  it was not re-executed end-to-end. It requires the BodyM images from S3 plus
  long preprocessing and nested-CV runs; see `PROJECT_INVENTORY.md` for the
  exact remaining work.

## Project Inventory
See `PROJECT_INVENTORY.md` for the complete artifact inventory, recovery
provenance and remaining regeneration steps.
"""

with open(
    os.path.join(PROJECT_ROOT, "README.md"),
    "w",
    encoding="utf-8",
    newline="\n",
) as f:
    f.write(readme.strip() + "\n")

print("README.md created!")


README.md created!


In [6]:
import os
import shutil
import subprocess
import sys

# `requirements.txt` documents the environment that produced the current run.
# Keep the original Colab freeze in `reference/` before it can be overwritten.
REFERENCE_DIR = os.path.join(PROJECT_ROOT, "reference")
os.makedirs(REFERENCE_DIR, exist_ok=True)

requirements_path = os.path.join(PROJECT_ROOT, "requirements.txt")
archive_path = os.path.join(
    REFERENCE_DIR, "requirements-colab-original.txt"
)

if os.path.isfile(requirements_path) and not os.path.isfile(archive_path):
    shutil.copy2(requirements_path, archive_path)
    print("Archived existing freeze:", archive_path)

result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    check=True
)

with open(
    requirements_path,
    "w",
    encoding="utf-8",
    newline="\n",
) as f:
    f.write(result.stdout)

print("requirements.txt saved!")
print("Original Colab freeze archive:", archive_path)


requirements.txt saved!
Original Colab freeze archive: d:\DorjiBari-Research\reference\requirements-colab-original.txt


In [5]:

import sys
import os
import platform
import torch
import cv2
import numpy as np
import pandas as pd
import sklearn

report = f"""
DORJIBARI RESEARCH - ENVIRONMENT REPORT

Python: {sys.version}
Platform: {platform.platform()}
PyTorch: {torch.__version__}
OpenCV: {cv2.__version__}
NumPy: {np.__version__}
Pandas: {pd.__version__}
Scikit-learn: {sklearn.__version__}

GPU Available: {torch.cuda.is_available()}
GPU Name: {
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else "Not Available"
}
CUDA Version: {torch.version.cuda}
"""

# Keep the original Colab report in `reference/` before replacing the
# top-level file with this run's environment.
REFERENCE_DIR = os.path.join(PROJECT_ROOT, "reference")
os.makedirs(REFERENCE_DIR, exist_ok=True)

report_path = os.path.join(
    PROJECT_ROOT, "environment_report.txt"
)
archive_path = os.path.join(
    REFERENCE_DIR, "environment_report-colab-original.txt"
)

if os.path.isfile(report_path) and not os.path.isfile(archive_path):
    import shutil
    shutil.copy2(report_path, archive_path)
    print("Archived existing environment report:", archive_path)

with open(
    report_path,
    "w",
    encoding="utf-8",
    newline="\n",
) as f:
    f.write(report)

print(report)
print("Environment report saved!")



DORJIBARI RESEARCH - ENVIRONMENT REPORT

Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26300-SP0
PyTorch: 2.15.0+cpu
OpenCV: 4.14.0
NumPy: 2.5.5
Pandas: 3.1.1
Scikit-learn: 1.9.1

GPU Available: False
GPU Name: Not Available
CUDA Version: None

Environment report saved!


In [ ]:

import os

required_files = [
    "README.md",
    "requirements.txt",
    "environment_report.txt",
]

print("DORJIBARI PROJECT VERIFICATION")
print("=" * 45)

for file in required_files:
    path = os.path.join(PROJECT_ROOT, file)
    print(f"{file}: {'EXISTS' if os.path.isfile(path) else 'MISSING'}")

print("\nProject Root:", PROJECT_ROOT)

print("\nReference archive:")
for file in [
    "requirements-colab-original.txt",
    "environment_report-colab-original.txt",
]:
    path = os.path.join(PROJECT_ROOT, "reference", file)
    print(f"{file}: {'EXISTS' if os.path.isfile(path) else 'MISSING'}")

print("PROJECT INITIALIZATION COMPLETED")


DORJIBARI PROJECT VERIFICATION
README.md: EXISTS
requirements.txt: EXISTS
environment_report.txt: EXISTS

Project Root: d:\DorjiBari-Research

Reference archive:
requirements-colab-original.txt: EXISTS
environment_report-colab-original.txt: EXISTS
PROJECT INITIALIZATION COMPLETED
